In [1]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [6]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [7]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [10]:
class VAEBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU(),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU()
        )

    def forward(self, x):
        return self.block(x)

In [11]:
class VAEEncoder3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.enc1 = VAEBlock3D(
            in_channels,
            base_channels
        )

        self.down1 = nn.Conv3d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.enc2 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.down2 = nn.Conv3d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.bottleneck = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.to_mu = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )

        self.to_logvar = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )

    def forward(self, x):

        x = self.enc1(x)
        x = self.down1(x)

        x = self.enc2(x)
        x = self.down2(x)

        x = self.bottleneck(x)

        mu = self.to_mu(x)
        logvar = self.to_logvar(x)

        return mu, logvar

In [12]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [13]:
class VAEDecoder3D(nn.Module):
    def __init__(
        self,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.from_latent = nn.Conv3d(
            latent_channels,
            base_channels * 4,
            kernel_size=3,
            padding=1
        )

        self.dec2 = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.up2 = nn.ConvTranspose3d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.up1 = nn.ConvTranspose3d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.final_block = VAEBlock3D(
            base_channels,
            base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, z):

        x = self.from_latent(z)

        x = self.dec2(x)
        x = self.up2(x)

        x = self.dec1(x)
        x = self.up1(x)

        x = self.final_block(x)
        x = self.output_conv(x)

        x = torch.sigmoid(x)

        return x

In [14]:
class VAE3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.encoder = VAEEncoder3D(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

        self.decoder = VAEDecoder3D(
            out_channels=out_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

    def forward(self, x):
        mu, logvar = self.encoder(x)

        z = reparameterize(
            mu,
            logvar
        )

        reconstruction = self.decoder(z)

        return reconstruction, mu, logvar, z

In [15]:
def vae_loss_v3(
    reconstruction,
    target,
    mu,
    logvar,
    kl_weight=1e-6,
    multiscale_weight=0.1
):
    # Full-resolution L1 reconstruction loss
    recon_loss = F.l1_loss(
        reconstruction,
        target
    )

    # Half-resolution structural loss
    recon_half = F.avg_pool3d(
        reconstruction,
        kernel_size=2,
        stride=2
    )

    target_half = F.avg_pool3d(
        target,
        kernel_size=2,
        stride=2
    )

    # Quarter-resolution structural loss
    recon_quarter = F.avg_pool3d(
        reconstruction,
        kernel_size=4,
        stride=4
    )

    target_quarter = F.avg_pool3d(
        target,
        kernel_size=4,
        stride=4
    )

    half_loss = F.l1_loss(
        recon_half,
        target_half
    )

    quarter_loss = F.l1_loss(
        recon_quarter,
        target_quarter
    )

    multiscale_loss = (
        0.5 * half_loss
        + 0.5 * quarter_loss
    )

    # KL divergence
    kl_loss = -0.5 * torch.mean(
        1
        + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    # Total loss
    total_loss = (
        recon_loss
        + multiscale_weight * multiscale_loss
        + kl_weight * kl_loss
    )

    return (
        total_loss,
        recon_loss,
        kl_loss,
        multiscale_loss
    )

In [16]:
def train_vae_v3(
    model,
    train_loader,
    epochs,
    optimizer,
    device,
    checkpoint_dir="vae_x4_v3_checkpoints",
    kl_weight=1e-6,
    multiscale_weight=0.1,
    start_epoch=10
):
    import os

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    loss_history = []
    recon_history = []
    kl_history = []
    multiscale_history = []

    for local_epoch in range(epochs):

        # Continue numbering from the original VAE
        current_epoch = start_epoch + local_epoch + 1

        model.train()

        epoch_loss = 0.0
        epoch_recon = 0.0
        epoch_kl = 0.0
        epoch_multiscale = 0.0

        for batch_idx, batch in enumerate(train_loader):

            x = batch["image"].to(device)

            optimizer.zero_grad()

            reconstruction, mu, logvar, z = model(x)

            (
                loss,
                recon_loss,
                kl_loss,
                multiscale_loss
            ) = vae_loss_v3(
                reconstruction=reconstruction,
                target=x,
                mu=mu,
                logvar=logvar,
                kl_weight=kl_weight,
                multiscale_weight=multiscale_weight
            )

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()
            epoch_multiscale += multiscale_loss.item()

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Epoch {current_epoch} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss: {loss.item():.6f} | "
                    f"Recon: {recon_loss.item():.6f} | "
                    f"KL: {kl_loss.item():.6f} | "
                    f"Multi-scale: {multiscale_loss.item():.6f}"
                )

        avg_loss = (
            epoch_loss / len(train_loader)
        )

        avg_recon = (
            epoch_recon / len(train_loader)
        )

        avg_kl = (
            epoch_kl / len(train_loader)
        )

        avg_multiscale = (
            epoch_multiscale / len(train_loader)
        )

        loss_history.append(avg_loss)
        recon_history.append(avg_recon)
        kl_history.append(avg_kl)
        multiscale_history.append(avg_multiscale)

        print(
            f"Epoch {current_epoch} completed | "
            f"Loss: {avg_loss:.6f} | "
            f"Recon: {avg_recon:.6f} | "
            f"KL: {avg_kl:.6f} | "
            f"Multi-scale: {avg_multiscale:.6f}"
        )

        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"vae_v3_epoch_{current_epoch:03d}.pt"
        )

        torch.save(
            {
                "epoch": current_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
                "recon_loss": avg_recon,
                "kl_loss": avg_kl,
                "multiscale_loss": avg_multiscale,
                "kl_weight": kl_weight,
                "multiscale_weight": multiscale_weight
            },
            checkpoint_path
        )

        print(
            "Saved:",
            checkpoint_path
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v3_loss_history.npy"
            ),
            np.array(loss_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v3_recon_history.npy"
            ),
            np.array(recon_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v3_kl_history.npy"
            ),
            np.array(kl_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v3_multiscale_history.npy"
            ),
            np.array(multiscale_history)
        )

    return (
        loss_history,
        recon_history,
        kl_history,
        multiscale_history
    )

In [17]:
def load_vae_checkpoint(
    model,
    optimizer,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

    loaded_epoch = checkpoint["epoch"]

    print(
        f"Loaded VAE checkpoint from epoch {loaded_epoch}"
    )

    return loaded_epoch

In [18]:
@torch.no_grad()
def reconstruct_vae(
    model,
    image,
    device
):
    model.eval()

    image = image.to(device)

    reconstruction, mu, logvar, z = model(image)

    return reconstruction

In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

vae = VAE3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    latent_channels=4
).to(device)

vae_checkpoint = torch.load(
    "vae_x4_v3_checkpoints/vae_v3_epoch_015.pt",
    map_location=device
)

vae.load_state_dict(
    vae_checkpoint["model_state_dict"]
)

vae.eval()

for p in vae.parameters():
    p.requires_grad = False

print("Device:", device)
print("Loaded frozen VAE epoch:", vae_checkpoint["epoch"])
print(
    "VAE frozen:",
    all(not p.requires_grad for p in vae.parameters())
)

Device: cuda
Loaded frozen VAE epoch: 15
VAE frozen: True


In [20]:
latent_stats_path = "conditional_ldm_v4_latent_stats.npz"

if os.path.exists(latent_stats_path):

    stats = np.load(latent_stats_path)

    LATENT_MEAN = torch.tensor(
        stats["mean"],
        dtype=torch.float32
    ).view(1, 4, 1, 1, 1)

    LATENT_STD = torch.tensor(
        stats["std"],
        dtype=torch.float32
    ).view(1, 4, 1, 1, 1)

    print("Loaded cached latent statistics.")

else:

    print("Computing latent statistics...")

    channel_sum = torch.zeros(
        4,
        dtype=torch.float64
    )

    channel_sq_sum = torch.zeros(
        4,
        dtype=torch.float64
    )

    voxel_count = 0

    vae.eval()

    with torch.no_grad():

        for i in range(len(train_dataset)):

            x = (
                train_dataset[i]["image"]
                .unsqueeze(0)
                .to(device)
            )

            mu, logvar = vae.encoder(x)

            # Deterministic latent for diffusion
            z = mu

            z_cpu = (
                z.detach()
                .cpu()
                .double()
            )

            channel_sum += z_cpu.sum(
                dim=(0, 2, 3, 4)
            )

            channel_sq_sum += (
                z_cpu ** 2
            ).sum(
                dim=(0, 2, 3, 4)
            )

            voxel_count += (
                z.shape[0]
                * z.shape[2]
                * z.shape[3]
                * z.shape[4]
            )

            if (i + 1) % 100 == 0:
                print(
                    f"Processed {i + 1}/"
                    f"{len(train_dataset)}"
                )

    mean = channel_sum / voxel_count

    variance = (
        channel_sq_sum / voxel_count
        - mean ** 2
    )

    std = torch.sqrt(
        torch.clamp(
            variance,
            min=1e-12
        )
    )

    LATENT_MEAN = (
        mean.float()
        .view(1, 4, 1, 1, 1)
    )

    LATENT_STD = (
        std.float()
        .view(1, 4, 1, 1, 1)
    )

    np.savez(
        latent_stats_path,
        mean=mean.numpy(),
        std=std.numpy()
    )

    print(
        "Saved:",
        latent_stats_path
    )


print(
    "LATENT_MEAN:",
    LATENT_MEAN.flatten()
)

print(
    "LATENT_STD:",
    LATENT_STD.flatten()
)

assert torch.all(
    LATENT_STD > 0
)

Computing latent statistics...
Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000
Saved: conditional_ldm_v4_latent_stats.npz
LATENT_MEAN: tensor([-0.0313, -0.1939,  0.1352,  0.0145])
LATENT_STD: tensor([0.6804, 0.9484, 1.4222, 0.2118])


In [21]:
entropy_stats_path = (
    "conditional_ldm_v4_entropy_stats.npz"
)

if os.path.exists(
    entropy_stats_path
):

    stats = np.load(
        entropy_stats_path
    )

    ENTROPY_MEAN = float(
        stats["mean"]
    )

    ENTROPY_STD = float(
        stats["std"]
    )

    print(
        "Loaded cached entropy statistics."
    )

else:

    print(
        "Computing entropy statistics..."
    )

    entropy_values = []

    for i in range(
        len(train_dataset)
    ):

        entropy_values.append(
            train_dataset[i][
                "heterogeneity"
            ].item()
        )

    entropy_values = np.asarray(
        entropy_values,
        dtype=np.float32
    )

    ENTROPY_MEAN = float(
        entropy_values.mean()
    )

    ENTROPY_STD = float(
        entropy_values.std()
    )

    np.savez(
        entropy_stats_path,
        mean=ENTROPY_MEAN,
        std=ENTROPY_STD
    )


print(
    "Entropy mean:",
    ENTROPY_MEAN
)

print(
    "Entropy std:",
    ENTROPY_STD
)

assert ENTROPY_STD > 0

Computing entropy statistics...
Entropy mean: 6.870205879211426
Entropy std: 0.33203670382499695


In [23]:
import math

timesteps = 1000


def cosine_beta_schedule(
    timesteps,
    s=0.008
):

    steps = timesteps + 1

    x = torch.linspace(
        0,
        timesteps,
        steps,
        dtype=torch.float64
    )

    alpha_bar = torch.cos(
        (
            (x / timesteps + s)
            / (1.0 + s)
        )
        * math.pi
        * 0.5
    ) ** 2

    alpha_bar = (
        alpha_bar
        / alpha_bar[0]
    )

    betas = (
        1.0
        - alpha_bar[1:]
        / alpha_bar[:-1]
    )

    return torch.clamp(
        betas,
        min=1e-8,
        max=0.999
    ).float()


def rescale_zero_terminal_snr(
    betas
):

    alphas = (
        1.0 - betas
    )

    alpha_bar = torch.cumprod(
        alphas,
        dim=0
    )

    sqrt_alpha_bar = torch.sqrt(
        alpha_bar
    )

    first = sqrt_alpha_bar[0].clone()
    last = sqrt_alpha_bar[-1].clone()

    sqrt_alpha_bar = (
        sqrt_alpha_bar - last
    )

    sqrt_alpha_bar = (
        sqrt_alpha_bar
        * first
        / (first - last)
    )

    alpha_bar = (
        sqrt_alpha_bar ** 2
    )

    new_alphas = (
        alpha_bar[1:]
        / alpha_bar[:-1]
    )

    new_alphas = torch.cat(
        [
            alpha_bar[:1],
            new_alphas
        ]
    )

    return (
        1.0 - new_alphas
    ).float()


betas = cosine_beta_schedule(
    timesteps
)

betas = rescale_zero_terminal_snr(
    betas
)

alphas = (
    1.0 - betas
)

alphas_cumprod = torch.cumprod(
    alphas,
    dim=0
)

alphas_cumprod_prev = F.pad(
    alphas_cumprod[:-1],
    (1, 0),
    value=1.0
)

sqrt_alphas_cumprod = torch.sqrt(
    alphas_cumprod
)

sqrt_one_minus_alphas_cumprod = torch.sqrt(
    1.0 - alphas_cumprod
)

posterior_variance = (
    betas
    * (
        1.0
        - alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_variance = torch.clamp(
    posterior_variance,
    min=1e-20
)

posterior_mean_coef1 = (
    betas
    * torch.sqrt(
        alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_mean_coef2 = (
    (
        1.0
        - alphas_cumprod_prev
    )
    * torch.sqrt(
        alphas
    )
    / (
        1.0
        - alphas_cumprod
    )
)


final_snr = (
    alphas_cumprod[-1]
    / torch.clamp(
        1.0
        - alphas_cumprod[-1],
        min=1e-12
    )
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

print(
    "Final SNR:",
    final_snr.item()
)

assert (
    alphas_cumprod[-1].item()
    == 0.0
)

print(
    "Zero-terminal-SNR check passed."
)

Final alpha_cumprod: 0.0
Final SNR: 0.0
Zero-terminal-SNR check passed.


In [24]:
def q_sample(
    x0,
    t,
    noise=None
):

    if noise is None:
        noise = torch.randn_like(x0)

    a = (
        sqrt_alphas_cumprod
        .to(x0.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    b = (
        sqrt_one_minus_alphas_cumprod
        .to(x0.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    return (
        a * x0
        + b * noise
    )


def get_v_target(
    x0,
    noise,
    t
):

    a = (
        sqrt_alphas_cumprod
        .to(x0.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    b = (
        sqrt_one_minus_alphas_cumprod
        .to(x0.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    return (
        a * noise
        - b * x0
    )


def v_to_x0(
    xt,
    v,
    t
):

    a = (
        sqrt_alphas_cumprod
        .to(xt.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    b = (
        sqrt_one_minus_alphas_cumprod
        .to(xt.device)[t]
        .view(-1, 1, 1, 1, 1)
    )

    return (
        a * xt
        - b * v
    )


def prepare_latent_mask(
    mask
):

    mask = (
        mask.squeeze(1)
        .long()
    )

    onehot = F.one_hot(
        mask,
        num_classes=4
    )

    onehot = (
        onehot
        .permute(
            0,
            4,
            1,
            2,
            3
        )
        .float()
    )

    # Remove background channel
    onehot = onehot[:, 1:]

    onehot = F.interpolate(
        onehot,
        size=(52, 56, 40),
        mode="nearest"
    )

    return onehot

In [25]:
class SinusoidalTimeEmbedding(nn.Module):

    def __init__(
        self,
        dim
    ):
        super().__init__()
        self.dim = dim

    def forward(
        self,
        t
    ):

        half = (
            self.dim // 2
        )

        scale = (
            math.log(10000)
            / (half - 1)
        )

        emb = torch.exp(
            torch.arange(
                half,
                device=t.device
            )
            * -scale
        )

        emb = (
            t[:, None].float()
            * emb[None, :]
        )

        return torch.cat(
            [
                emb.sin(),
                emb.cos()
            ],
            dim=1
        )


class ResBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        condition_dim,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.GroupNorm(
            8,
            in_channels
        )

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.condition_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                out_channels * 2
            )
        )

        nn.init.zeros_(
            self.condition_mlp[-1].weight
        )

        nn.init.zeros_(
            self.condition_mlp[-1].bias
        )

        self.norm2 = nn.GroupNorm(
            8,
            out_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.conv2.weight
        )

        nn.init.zeros_(
            self.conv2.bias
        )

        if (
            in_channels
            != out_channels
        ):

            self.skip = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )

        else:
            self.skip = nn.Identity()


    def forward(
        self,
        x,
        condition
    ):

        residual = self.skip(x)

        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)

        scale, shift = (
            self.condition_mlp(
                condition
            )
            .chunk(
                2,
                dim=1
            )
        )

        scale = scale[
            :, :, None, None, None
        ]

        shift = shift[
            :, :, None, None, None
        ]

        h = self.norm2(h)

        h = (
            h
            * (1.0 + scale)
            + shift
        )

        h = F.silu(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return (
            residual + h
        )


class SelfAttention3D(nn.Module):

    def __init__(
        self,
        channels,
        heads=8
    ):
        super().__init__()

        self.norm = nn.GroupNorm(
            8,
            channels
        )

        self.attention = (
            nn.MultiheadAttention(
                embed_dim=channels,
                num_heads=heads,
                batch_first=True
            )
        )


    def forward(
        self,
        x
    ):

        b, c, d, h, w = x.shape

        residual = x

        x = self.norm(x)

        x = (
            x.permute(
                0, 2, 3, 4, 1
            )
            .reshape(
                b,
                d * h * w,
                c
            )
        )

        x, _ = self.attention(
            x,
            x,
            x,
            need_weights=False
        )

        x = (
            x.reshape(
                b,
                d,
                h,
                w,
                c
            )
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .contiguous()
        )

        return (
            residual + x
        )

In [26]:
class ConditionalLatentUNet3D(nn.Module):

    def __init__(
        self,
        latent_channels=4,
        base_channels=64,
        condition_dim=256,
        entropy_scale=0.1
    ):
        super().__init__()

        self.entropy_scale = (
            entropy_scale
        )

        # ============================
        # Time condition
        # ============================

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(
                condition_dim
            ),
            nn.Linear(
                condition_dim,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        # ============================
        # Entropy condition
        # ============================

        self.entropy_embedding = nn.Sequential(
            nn.Linear(
                1,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].weight
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].bias
        )

        # ============================
        # Latent input
        # ============================

        self.input_conv = nn.Conv3d(
            latent_channels,
            64,
            kernel_size=3,
            padding=1
        )

        # ============================
        # Mask projections
        # ============================

        self.mask_level1 = nn.Conv3d(
            3,
            64,
            kernel_size=3,
            padding=1
        )

        self.mask_level2 = nn.Conv3d(
            3,
            128,
            kernel_size=3,
            padding=1
        )

        self.mask_level3 = nn.Conv3d(
            3,
            256,
            kernel_size=3,
            padding=1
        )

        # Start mask influence softly
        nn.init.zeros_(
            self.mask_level1.weight
        )
        nn.init.zeros_(
            self.mask_level1.bias
        )

        nn.init.zeros_(
            self.mask_level2.weight
        )
        nn.init.zeros_(
            self.mask_level2.bias
        )

        nn.init.zeros_(
            self.mask_level3.weight
        )
        nn.init.zeros_(
            self.mask_level3.bias
        )

        # ============================
        # Encoder level 1
        # 52 x 56 x 40
        # ============================

        self.enc1a = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.enc1b = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.down1 = nn.Conv3d(
            64,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        # ============================
        # Encoder level 2
        # 26 x 28 x 20
        # ============================

        self.enc2a = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.enc2b = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.down2 = nn.Conv3d(
            128,
            256,
            kernel_size=4,
            stride=2,
            padding=1
        )

        # ============================
        # Bottleneck
        # 13 x 14 x 10
        # ============================

        self.mid1 = ResBlock3D(
            256,
            256,
            condition_dim
        )

        self.mid_attention = (
            SelfAttention3D(
                256,
                heads=8
            )
        )

        self.mid2 = ResBlock3D(
            256,
            256,
            condition_dim
        )

        # ============================
        # Decoder
        # ============================

        self.up2 = nn.ConvTranspose3d(
            256,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec2a = ResBlock3D(
            256,
            128,
            condition_dim
        )

        self.dec2b = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.up1 = nn.ConvTranspose3d(
            128,
            64,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1a = ResBlock3D(
            128,
            64,
            condition_dim
        )

        self.dec1b = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.out_norm = nn.GroupNorm(
            8,
            64
        )

        self.out_conv = nn.Conv3d(
            64,
            latent_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.out_conv.weight
        )

        nn.init.zeros_(
            self.out_conv.bias
        )


    def forward(
        self,
        z,
        t,
        latent_mask,
        entropy
    ):

        # ============================
        # Global condition
        # ============================

        time_emb = self.time_embedding(
            t
        )

        entropy = (
            entropy
            .float()
            .view(-1, 1)
        )

        entropy_emb = (
            self.entropy_embedding(
                entropy
            )
        )

        condition = (
            time_emb
            +
            self.entropy_scale
            * entropy_emb
        )

        # ============================
        # Level 1 mask
        # ============================

        x = self.input_conv(z)

        mask1 = (
            0.25
            * torch.tanh(
                self.mask_level1(
                    latent_mask
                )
            )
        )

        x = x + mask1

        x = self.enc1a(
            x,
            condition
        )

        x = self.enc1b(
            x,
            condition
        )

        skip1 = x

        # ============================
        # Level 2
        # ============================

        x = self.down1(x)

        latent_mask2 = F.interpolate(
            latent_mask,
            size=x.shape[2:],
            mode="nearest"
        )

        mask2 = (
            0.20
            * torch.tanh(
                self.mask_level2(
                    latent_mask2
                )
            )
        )

        x = x + mask2

        x = self.enc2a(
            x,
            condition
        )

        x = self.enc2b(
            x,
            condition
        )

        skip2 = x

        # ============================
        # Bottleneck
        # ============================

        x = self.down2(x)

        latent_mask3 = F.interpolate(
            latent_mask,
            size=x.shape[2:],
            mode="nearest"
        )

        mask3 = (
            0.15
            * torch.tanh(
                self.mask_level3(
                    latent_mask3
                )
            )
        )

        x = x + mask3

        x = self.mid1(
            x,
            condition
        )

        x = self.mid_attention(x)

        x = self.mid2(
            x,
            condition
        )

        # ============================
        # Decoder level 2
        # ============================

        x = self.up2(x)

        assert (
            x.shape[2:]
            == skip2.shape[2:]
        )

        x = torch.cat(
            [
                x,
                skip2
            ],
            dim=1
        )

        x = self.dec2a(
            x,
            condition
        )

        x = self.dec2b(
            x,
            condition
        )

        # ============================
        # Decoder level 1
        # ============================

        x = self.up1(x)

        assert (
            x.shape[2:]
            == skip1.shape[2:]
        )

        x = torch.cat(
            [
                x,
                skip1
            ],
            dim=1
        )

        x = self.dec1a(
            x,
            condition
        )

        x = self.dec1b(
            x,
            condition
        )

        x = F.silu(
            self.out_norm(x)
        )

        return self.out_conv(x)

In [29]:
import copy

class EMA:

    def __init__(
        self,
        model,
        decay=0.9999
    ):

        self.decay = decay

        self.ema_model = (
            copy.deepcopy(model)
        )

        self.ema_model.eval()

        for p in (
            self.ema_model.parameters()
        ):
            p.requires_grad = False


    @torch.no_grad()
    def update(
        self,
        model
    ):

        ema_params = dict(
            self.ema_model.named_parameters()
        )

        model_params = dict(
            model.named_parameters()
        )

        for name, p in (
            model_params.items()
        ):

            ema_params[name].mul_(
                self.decay
            ).add_(
                p,
                alpha=(
                    1.0
                    - self.decay
                )
            )


def save_ldm_checkpoint(
    model,
    ema,
    optimizer,
    epoch,
    path
):

    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "ema_state_dict":
                ema.ema_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "latent_mean":
                LATENT_MEAN,

            "latent_std":
                LATENT_STD,

            "entropy_mean":
                ENTROPY_MEAN,

            "entropy_std":
                ENTROPY_STD
        },
        path
    )


def load_ldm_checkpoint(
    model,
    ema,
    optimizer,
    path,
    device
):

    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    ema.ema_model.load_state_dict(
        checkpoint[
            "ema_state_dict"
        ]
    )

    if optimizer is not None:

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

    return checkpoint[
        "epoch"
    ]

In [30]:
model = ConditionalLatentUNet3D(
    latent_channels=4,
    base_channels=64,
    condition_dim=256,
    entropy_scale=0.1
).to(device)

ema = EMA(
    model,
    decay=0.9999
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Conditional LDM V4 parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Conditional LDM V4 ready."
)

Conditional LDM V4 parameters: 18,517,444
Trainable parameters: 18,517,444
Conditional LDM V4 ready.


In [31]:
def train_conditional_ldm(
    model,
    ema,
    vae,
    train_loader,
    optimizer,
    device,
    epochs=50,
    checkpoint_dir=(
        "conditional_ldm_v4_checkpoints"
    ),
    tumour_lambda=0.25,
    condition_dropout_prob=0.15,
    entropy_dropout_prob=0.15
):

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    history = []

    latent_mean = (
        LATENT_MEAN
        .to(device)
    )

    latent_std = (
        LATENT_STD
        .to(device)
    )

    use_amp = (
        device.type == "cuda"
    )

    scaler = (
        torch.cuda.amp
        .GradScaler(
            enabled=use_amp
        )
    )

    vae.eval()


    for epoch in range(
        epochs
    ):

        model.train()

        epoch_total = 0.0
        epoch_global = 0.0
        epoch_tumour = 0.0


        for batch_idx, batch in enumerate(
            train_loader
        ):

            x = (
                batch["image"]
                .to(
                    device,
                    non_blocking=True
                )
            )

            mask = (
                batch["mask"]
                .to(
                    device,
                    non_blocking=True
                )
            )

            entropy = (
                batch[
                    "heterogeneity"
                ]
                .to(
                    device,
                    non_blocking=True
                )
            )

            entropy = (
                entropy
                - ENTROPY_MEAN
            ) / ENTROPY_STD


            # ========================
            # Frozen VAE encode
            # ========================

            with torch.no_grad():

                mu, logvar = (
                    vae.encoder(x)
                )

                z = mu

                z = (
                    z - latent_mean
                ) / latent_std


            latent_mask = (
                prepare_latent_mask(
                    mask
                )
            )


            # ========================
            # Condition dropout
            # ========================

            cond_mask = (
                latent_mask.clone()
            )

            cond_entropy = (
                entropy.clone()
            )

            random_value = (
                torch.rand(1).item()
            )

            if (
                random_value
                < condition_dropout_prob
            ):

                cond_mask = (
                    torch.zeros_like(
                        cond_mask
                    )
                )

                cond_entropy = (
                    torch.zeros_like(
                        cond_entropy
                    )
                )

            elif (
                random_value
                <
                condition_dropout_prob
                + entropy_dropout_prob
            ):

                cond_entropy = (
                    torch.zeros_like(
                        cond_entropy
                    )
                )


            # ========================
            # Diffusion
            # ========================

            t = torch.randint(
                low=0,
                high=timesteps,
                size=(z.shape[0],),
                device=device,
                dtype=torch.long
            )

            noise = torch.randn_like(z)

            z_t = q_sample(
                z,
                t,
                noise
            )

            v_target = get_v_target(
                z,
                noise,
                t
            )

            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                v_pred = model(
                    z_t,
                    t,
                    cond_mask,
                    cond_entropy
                )

                squared_error = (
                    v_pred
                    - v_target
                ) ** 2

                global_loss = (
                    squared_error.mean()
                )

                tumour_region = (
                    latent_mask.sum(
                        dim=1,
                        keepdim=True
                    )
                    > 0
                )

                tumour_region = (
                    tumour_region.expand(
                        -1,
                        4,
                        -1,
                        -1,
                        -1
                    )
                )

                if tumour_region.any():

                    tumour_loss = (
                        squared_error[
                            tumour_region
                        ]
                        .mean()
                    )

                else:

                    tumour_loss = (
                        torch.zeros(
                            (),
                            device=device,
                            dtype=global_loss.dtype
                        )
                    )

                loss = (
                    global_loss
                    +
                    tumour_lambda
                    * tumour_loss
                )


            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            ema.update(model)


            epoch_total += (
                loss.item()
            )

            epoch_global += (
                global_loss.item()
            )

            epoch_tumour += (
                tumour_loss.item()
            )


            if (
                batch_idx + 1
            ) % 20 == 0:

                print(
                    f"Epoch "
                    f"{epoch + 1}/{epochs} | "
                    f"Batch "
                    f"{batch_idx + 1}/"
                    f"{len(train_loader)} | "
                    f"Total="
                    f"{loss.item():.5f} | "
                    f"Global="
                    f"{global_loss.item():.5f} | "
                    f"Tumour="
                    f"{tumour_loss.item():.5f}"
                )


        n = len(train_loader)

        avg_total = (
            epoch_total / n
        )

        avg_global = (
            epoch_global / n
        )

        avg_tumour = (
            epoch_tumour / n
        )


        history.append(
            [
                avg_total,
                avg_global,
                avg_tumour
            ]
        )


        print(
            f"\nEpoch "
            f"{epoch + 1} completed | "
            f"Total={avg_total:.6f} | "
            f"Global={avg_global:.6f} | "
            f"Tumour={avg_tumour:.6f}\n"
        )


        checkpoint_path = os.path.join(
            checkpoint_dir,
            (
                f"conditional_ldm_v4_"
                f"epoch_{epoch + 1:03d}.pt"
            )
        )


        save_ldm_checkpoint(
            model=model,
            ema=ema,
            optimizer=optimizer,
            epoch=epoch + 1,
            path=checkpoint_path
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "conditional_ldm_v4_loss_history.npy"
            ),
            np.asarray(
                history,
                dtype=np.float32
            )
        )


        print(
            "Saved:",
            checkpoint_path
        )

In [32]:
@torch.no_grad()
def sample_conditional_ldm(
    model,
    vae,
    mask,
    entropy,
    device,
    seed=42
):

    model.eval()
    vae.eval()

    torch.manual_seed(seed)

    latent_mean = (
        LATENT_MEAN
        .to(device)
    )

    latent_std = (
        LATENT_STD
        .to(device)
    )

    latent_mask = (
        prepare_latent_mask(
            mask.to(device)
        )
    )

    entropy = (
        entropy
        .to(device)
        .float()
    )

    entropy = (
        entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    x = torch.randn(
        (
            mask.shape[0],
            4,
            52,
            56,
            40
        ),
        device=device
    )


    coef1 = (
        posterior_mean_coef1
        .to(device)
    )

    coef2 = (
        posterior_mean_coef2
        .to(device)
    )

    posterior_var = (
        posterior_variance
        .to(device)
    )


    for step in reversed(
        range(timesteps)
    ):

        t = torch.full(
            (x.shape[0],),
            step,
            device=device,
            dtype=torch.long
        )

        v_pred = model(
            x,
            t,
            latent_mask,
            entropy
        )

        x0_pred = v_to_x0(
            x,
            v_pred,
            t
        )

        # Avoid exploding latent prediction
        x0_pred = torch.clamp(
            x0_pred,
            -5.0,
            5.0
        )

        model_mean = (
            coef1[step]
            * x0_pred
            +
            coef2[step]
            * x
        )

        if step > 0:

            noise = torch.randn_like(x)

            x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[step]
                )
                * noise
            )

        else:

            x = model_mean


    # Return to original VAE latent distribution
    generated_latent = (
        x * latent_std
        + latent_mean
    )

    generated_image = (
        vae.decoder(
            generated_latent
        )
    )

    return (
        generated_image,
        generated_latent
    )

In [ ]:
train_conditional_ldm(
    model=model,
    ema=ema,
    vae=vae,
    train_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epochs=50,
    checkpoint_dir=(
        "conditional_ldm_v4_checkpoints"
    ),
    tumour_lambda=0.25,
    condition_dropout_prob=0.15,
    entropy_dropout_prob=0.15
)

In [ ]:
# ============================================================
# Conditional LDM V4 - Final Sampling
# ============================================================

final_checkpoint = (
    "conditional_ldm_v4_checkpoints/"
    "conditional_ldm_v4_epoch_050.pt"
)

loaded_epoch = load_ldm_checkpoint(
    model=model,
    ema=ema,
    optimizer=optimizer,
    path=final_checkpoint,
    device=device
)

print(
    "Loaded final epoch:",
    loaded_epoch
)


sample = train_dataset[0]

real_image = (
    sample["image"]
    .unsqueeze(0)
    .to(device)
)

mask = (
    sample["mask"]
    .unsqueeze(0)
    .to(device)
)

entropy = (
    sample["heterogeneity"]
    .unsqueeze(0)
    .to(device)
)


print(
    "Subject:",
    sample["subject"]
)

print(
    "Raw entropy:",
    entropy.item()
)


generated_image, generated_latent = (
    sample_conditional_ldm(
        model=ema.ema_model,
        vae=vae,
        mask=mask,
        entropy=entropy,
        device=device,
        seed=42
    )
)


print(
    "Generated image shape:",
    generated_image.shape
)

print(
    "Generated latent shape:",
    generated_latent.shape
)

print(
    "Generated range:",
    generated_image.min().item(),
    generated_image.max().item()
)


real_np = (
    real_image[
        0,
        0
    ]
    .detach()
    .cpu()
    .numpy()
)

generated_np = (
    generated_image[
        0,
        0
    ]
    .detach()
    .cpu()
    .numpy()
)

mask_np = (
    mask[
        0,
        0
    ]
    .detach()
    .cpu()
    .numpy()
)


# VAE V3 uses [0,1]
real_display = np.clip(
    real_np,
    0.0,
    1.0
)

generated_display = np.clip(
    generated_np,
    0.0,
    1.0
)


tumour_per_slice = (
    mask_np > 0
).sum(
    axis=(0, 1)
)

tumour_slice = int(
    np.argmax(
        tumour_per_slice
    )
)


plt.figure(
    figsize=(15, 5)
)


plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    real_display[
        :,
        :,
        tumour_slice
    ],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Real T2f"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    generated_display[
        :,
        :,
        tumour_slice
    ],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Conditional LDM V4"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    mask_np[
        :,
        :,
        tumour_slice
    ],
    cmap="viridis"
)

plt.title(
    "Tumour Condition"
)

plt.axis("off")


plt.tight_layout()
plt.show()


# ============================
# Orthogonal views
# ============================

x_mid = (
    generated_display.shape[0]
    // 2
)

y_mid = (
    generated_display.shape[1]
    // 2
)

z_mid = (
    generated_display.shape[2]
    // 2
)


plt.figure(
    figsize=(12, 4)
)


plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    generated_display[
        x_mid,
        :,
        :
    ].T,
    cmap="gray",
    origin="lower",
    vmin=0,
    vmax=1
)

plt.title(
    "Sagittal"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    generated_display[
        :,
        y_mid,
        :
    ].T,
    cmap="gray",
    origin="lower",
    vmin=0,
    vmax=1
)

plt.title(
    "Coronal"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    generated_display[
        :,
        :,
        z_mid
    ],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Axial"
)

plt.axis("off")


plt.tight_layout()
plt.show()


print()
print("Generated statistics")

print(
    "Min:",
    generated_np.min()
)

print(
    "Max:",
    generated_np.max()
)

print(
    "Mean:",
    generated_np.mean()
)

print(
    "Std:",
    generated_np.std()
)

In [33]:
# ============================================================
# Conditional LDM V4 - Single Training Step Smoke Test
# ============================================================

model.train()
vae.eval()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))

x = batch["image"].to(
    device,
    non_blocking=True
)

mask = batch["mask"].to(
    device,
    non_blocking=True
)

entropy = batch["heterogeneity"].to(
    device,
    non_blocking=True
)

entropy = (
    entropy
    - ENTROPY_MEAN
) / ENTROPY_STD


# ------------------------------------------------------------
# Frozen VAE encode
# ------------------------------------------------------------

with torch.no_grad():

    mu, logvar = vae.encoder(x)

    z = mu

    latent_mean = LATENT_MEAN.to(device)
    latent_std = LATENT_STD.to(device)

    z = (
        z
        - latent_mean
    ) / latent_std


latent_mask = prepare_latent_mask(
    mask
)


print("Input image shape:", x.shape)
print("Latent shape:", z.shape)
print("Latent mask shape:", latent_mask.shape)
print("Entropy shape:", entropy.shape)


assert z.shape[1:] == (
    4,
    52,
    56,
    40
)

assert latent_mask.shape[1:] == (
    3,
    52,
    56,
    40
)


# ------------------------------------------------------------
# Diffusion
# ------------------------------------------------------------

t = torch.randint(
    low=0,
    high=timesteps,
    size=(z.shape[0],),
    device=device,
    dtype=torch.long
)

noise = torch.randn_like(z)

z_t = q_sample(
    z,
    t,
    noise
)

v_target = get_v_target(
    z,
    noise,
    t
)


# ------------------------------------------------------------
# One real forward + backward
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

use_amp = (
    device.type == "cuda"
)


with torch.cuda.amp.autocast(
    enabled=use_amp
):

    v_pred = model(
        z_t,
        t,
        latent_mask,
        entropy
    )

    squared_error = (
        v_pred
        - v_target
    ) ** 2

    global_loss = (
        squared_error.mean()
    )

    tumour_region = (
        latent_mask.sum(
            dim=1,
            keepdim=True
        )
        > 0
    )

    tumour_region = (
        tumour_region.expand(
            -1,
            4,
            -1,
            -1,
            -1
        )
    )

    if tumour_region.any():

        tumour_loss = (
            squared_error[
                tumour_region
            ]
            .mean()
        )

    else:

        tumour_loss = torch.zeros(
            (),
            device=device,
            dtype=global_loss.dtype
        )

    loss = (
        global_loss
        + 0.25 * tumour_loss
    )


loss.backward()


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert (
    v_pred.shape
    == z.shape
)

assert torch.isfinite(
    loss
).item()

print()
print(
    "Conditional LDM V4 "
    "single training step passed."
)

print(
    "Loss:",
    loss.item()
)

print(
    "Global loss:",
    global_loss.item()
)

print(
    "Tumour loss:",
    tumour_loss.item()
)

print(
    "Predicted v shape:",
    v_pred.shape
)

print(
    "Latent timestep:",
    t.tolist()
)


if torch.cuda.is_available():

    print(
        "Current GPU allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Current GPU reserved:",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Peak GPU allocated:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Total GPU memory:",
        round(
            torch.cuda.get_device_properties(
                device
            ).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )


# ------------------------------------------------------------
# Clean up
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

del (
    batch,
    x,
    mask,
    entropy,
    mu,
    logvar,
    z,
    latent_mask,
    t,
    noise,
    z_t,
    v_target,
    v_pred,
    squared_error,
    tumour_region,
    global_loss,
    tumour_loss,
    loss
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Input image shape: torch.Size([1, 1, 208, 224, 160])
Latent shape: torch.Size([1, 4, 52, 56, 40])
Latent mask shape: torch.Size([1, 3, 52, 56, 40])
Entropy shape: torch.Size([1])


/tmp/ipykernel_3274405/262743195.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



Conditional LDM V4 single training step passed.
Loss: 1.312918782234192
Global loss: 1.000767707824707
Tumour loss: 1.248604416847229
Predicted v shape: torch.Size([1, 4, 52, 56, 40])
Latent timestep: [107]
Current GPU allocated: 0.41 GB
Current GPU reserved: 2.77 GB
Peak GPU allocated: 2.09 GB
Total GPU memory: 44.42 GB


In [34]:
# ============================================================
# CONDITIONAL LDM V4 — COMPLETE PRE-FLIGHT DIAGNOSTIC
# Run AFTER Cell 28 and BEFORE Cell 29
# ============================================================

print("=" * 72)
print("CONDITIONAL LDM V4 PRE-FLIGHT DIAGNOSTIC")
print("=" * 72)

model.train()
vae.eval()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


# ============================================================
# PART 1 — DIFFUSION SCHEDULE
# ============================================================

print("\n" + "=" * 72)
print("PART 1 — DIFFUSION SCHEDULE")
print("=" * 72)

print(
    "Timesteps:",
    timesteps
)

print(
    "First beta:",
    betas[0].item()
)

print(
    "Last beta:",
    betas[-1].item()
)

print(
    "First alpha_cumprod:",
    alphas_cumprod[0].item()
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

final_snr = (
    alphas_cumprod[-1]
    / torch.clamp(
        1.0 - alphas_cumprod[-1],
        min=1e-12
    )
)

print(
    "Final SNR:",
    final_snr.item()
)

assert torch.isfinite(
    betas
).all()

assert torch.isfinite(
    alphas_cumprod
).all()

assert (
    alphas_cumprod[-1].item()
    == 0.0
)

print(
    "RESULT: schedule check PASSED."
)


# ============================================================
# PART 2 — REAL SAMPLE + VAE LATENT
# ============================================================

print("\n" + "=" * 72)
print("PART 2 — VAE LATENT PIPELINE")
print("=" * 72)

batch = next(
    iter(train_loader)
)

x = batch["image"].to(
    device,
    non_blocking=True
)

mask = batch["mask"].to(
    device,
    non_blocking=True
)

entropy_raw = batch[
    "heterogeneity"
].to(
    device,
    non_blocking=True
)

entropy = (
    entropy_raw
    - ENTROPY_MEAN
) / ENTROPY_STD


with torch.no_grad():

    mu, logvar = vae.encoder(
        x
    )

    z_raw = mu

    latent_mean = (
        LATENT_MEAN
        .to(device)
    )

    latent_std = (
        LATENT_STD
        .to(device)
    )

    z = (
        z_raw
        - latent_mean
    ) / latent_std


latent_mask = prepare_latent_mask(
    mask
)


print(
    "Image shape:",
    tuple(x.shape)
)

print(
    "Raw latent shape:",
    tuple(z_raw.shape)
)

print(
    "Normalized latent shape:",
    tuple(z.shape)
)

print(
    "Latent mask shape:",
    tuple(latent_mask.shape)
)

print(
    "Raw entropy:",
    entropy_raw.item()
)

print(
    "Normalized entropy:",
    entropy.item()
)


assert z.shape[1:] == (
    4,
    52,
    56,
    40
)

assert latent_mask.shape[1:] == (
    3,
    52,
    56,
    40
)


print("\nRaw latent statistics")

for c in range(4):

    print(
        f"Channel {c}: "
        f"mean={z_raw[:, c].mean().item():.4f}, "
        f"std={z_raw[:, c].std().item():.4f}"
    )


print("\nNormalized latent statistics")

for c in range(4):

    print(
        f"Channel {c}: "
        f"mean={z[:, c].mean().item():.4f}, "
        f"std={z[:, c].std().item():.4f}"
    )


print(
    "\nTumour latent fraction:",
    (
        latent_mask.sum(
            dim=1
        ) > 0
    ).float().mean().item()
)


assert torch.isfinite(
    z
).all()

assert torch.isfinite(
    entropy
).all()

print(
    "RESULT: latent pipeline PASSED."
)


# ============================================================
# PART 3 — q_sample / v-PREDICTION ALGEBRA
# ============================================================

print("\n" + "=" * 72)
print("PART 3 — v-PREDICTION ALGEBRA")
print("=" * 72)

test_timesteps = [
    0,
    50,
    250,
    500,
    750,
    999
]

for test_t in test_timesteps:

    t_test = torch.full(
        (z.shape[0],),
        test_t,
        device=device,
        dtype=torch.long
    )

    noise_test = torch.randn_like(
        z
    )

    z_t_test = q_sample(
        z,
        t_test,
        noise_test
    )

    v_true = get_v_target(
        z,
        noise_test,
        t_test
    )

    z_recovered = v_to_x0(
        z_t_test,
        v_true,
        t_test
    )

    reconstruction_error = (
        torch.abs(
            z_recovered
            - z
        )
        .mean()
        .item()
    )

    print(
        f"t={test_t:3d} | "
        f"x0 algebra L1="
        f"{reconstruction_error:.8f}"
    )

    assert (
        reconstruction_error
        < 1e-4
    )


print(
    "RESULT: q/v mathematics PASSED."
)


# ============================================================
# PART 4 — CONDITION INPUT CHECK
# ============================================================

print("\n" + "=" * 72)
print("PART 4 — CONDITION INPUT CHECK")
print("=" * 72)

print(
    "Mask labels:",
    torch.unique(mask)
)

print(
    "Latent mask channel sums:",
    [
        latent_mask[:, c]
        .sum()
        .item()
        for c in range(3)
    ]
)

assert (
    latent_mask.sum()
    > 0
)

print(
    "Entropy finite:",
    torch.isfinite(
        entropy
    ).all().item()
)

print(
    "RESULT: conditions are valid."
)


# ============================================================
# PART 5 — REAL FORWARD + BACKWARD
# ============================================================

print("\n" + "=" * 72)
print("PART 5 — FORWARD / BACKWARD")
print("=" * 72)

t = torch.randint(
    low=0,
    high=timesteps,
    size=(z.shape[0],),
    device=device,
    dtype=torch.long
)

noise = torch.randn_like(
    z
)

z_t = q_sample(
    z,
    t,
    noise
)

v_target = get_v_target(
    z,
    noise,
    t
)


optimizer.zero_grad(
    set_to_none=True
)

use_amp = (
    device.type == "cuda"
)


with torch.cuda.amp.autocast(
    enabled=use_amp
):

    v_pred = model(
        z_t,
        t,
        latent_mask,
        entropy
    )

    squared_error = (
        v_pred
        - v_target
    ) ** 2

    global_loss = (
        squared_error.mean()
    )

    tumour_region = (
        latent_mask.sum(
            dim=1,
            keepdim=True
        )
        > 0
    )

    tumour_region = tumour_region.expand(
        -1,
        4,
        -1,
        -1,
        -1
    )

    if tumour_region.any():

        tumour_loss = (
            squared_error[
                tumour_region
            ]
            .mean()
        )

    else:

        tumour_loss = torch.zeros(
            (),
            device=device,
            dtype=global_loss.dtype
        )


    loss = (
        global_loss
        + 0.25
        * tumour_loss
    )


assert (
    v_pred.shape
    == z.shape
)

assert torch.isfinite(
    loss
).item()


loss.backward()


print(
    "Predicted v shape:",
    tuple(v_pred.shape)
)

print(
    "Random timestep:",
    t.tolist()
)

print(
    "Total loss:",
    loss.item()
)

print(
    "Global loss:",
    global_loss.item()
)

print(
    "Tumour loss:",
    tumour_loss.item()
)


# ============================================================
# PART 6 — GRADIENT FLOW
# ============================================================

print("\n" + "=" * 72)
print("PART 6 — GRADIENT FLOW")
print("=" * 72)


def grad_norm(parameter):

    if parameter.grad is None:
        return None

    return (
        parameter.grad
        .detach()
        .norm()
        .item()
    )


gradient_checks = {
    "input_conv":
        grad_norm(
            model.input_conv.weight
        ),

    "mask_level1":
        grad_norm(
            model.mask_level1.weight
        ),

    "mask_level2":
        grad_norm(
            model.mask_level2.weight
        ),

    "mask_level3":
        grad_norm(
            model.mask_level3.weight
        ),

    "out_conv":
        grad_norm(
            model.out_conv.weight
        )
}


for name, value in (
    gradient_checks.items()
):

    print(
        f"{name}:",
        value
    )


critical_gradients = [
    gradient_checks[
        "input_conv"
    ],
    gradient_checks[
        "mask_level1"
    ],
    gradient_checks[
        "out_conv"
    ]
]


if all(
    value is not None
    for value in critical_gradients
):

    print(
        "RESULT: critical gradient "
        "paths are connected."
    )

else:

    print(
        "WARNING: one or more "
        "critical gradients are missing."
    )


# ============================================================
# PART 7 — INITIAL CONDITION SENSITIVITY
# ============================================================

print("\n" + "=" * 72)
print("PART 7 — INITIAL CONDITION SENSITIVITY")
print("=" * 72)

model.eval()

with torch.no_grad():

    pred_correct = model(
        z_t,
        t,
        latent_mask,
        entropy
    )

    pred_zero_mask = model(
        z_t,
        t,
        torch.zeros_like(
            latent_mask
        ),
        entropy
    )

    pred_zero_entropy = model(
        z_t,
        t,
        latent_mask,
        torch.zeros_like(
            entropy
        )
    )


mask_difference = (
    torch.abs(
        pred_correct
        - pred_zero_mask
    )
    .mean()
    .item()
)

entropy_difference = (
    torch.abs(
        pred_correct
        - pred_zero_entropy
    )
    .mean()
    .item()
)


print(
    "Correct vs zero-mask difference:",
    mask_difference
)

print(
    "Correct vs zero-entropy difference:",
    entropy_difference
)

print(
    "NOTE: near-zero differences are EXPECTED "
    "before training because condition branches "
    "were deliberately zero-initialized."
)


# ============================================================
# PART 8 — GPU MEMORY
# ============================================================

print("\n" + "=" * 72)
print("PART 8 — GPU MEMORY")
print("=" * 72)

if torch.cuda.is_available():

    print(
        "Current allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Current reserved:",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Peak allocated:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Total GPU memory:",
        round(
            torch.cuda.get_device_properties(
                device
            ).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("AUTOMATIC PRE-FLIGHT SUMMARY")
print("=" * 72)

print(
    "✓ VAE latent shape correct."
)

print(
    "✓ Latent normalization finite."
)

print(
    "✓ Multi-class tumour mask valid."
)

print(
    "✓ Entropy condition finite."
)

print(
    "✓ Zero-terminal-SNR schedule valid."
)

print(
    "✓ q_sample / v-prediction mathematics valid."
)

print(
    "✓ Forward pass successful."
)

print(
    "✓ Backward pass successful."
)

print(
    "✓ Output latent shape correct."
)

print()
print(
    "IMPORTANT:"
)

print(
    "This test verifies the training pipeline, "
    "but cannot yet prove generation quality."
)

print(
    "After training begins, quality diagnostics "
    "can additionally test partial-noise latent "
    "reconstruction and condition sensitivity."
)

print("=" * 72)


# ============================================================
# CLEAN UP
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

model.train()

del (
    batch,
    x,
    mask,
    entropy_raw,
    entropy,
    mu,
    logvar,
    z_raw,
    z,
    latent_mask,
    t,
    noise,
    z_t,
    v_target,
    v_pred,
    squared_error,
    tumour_region,
    global_loss,
    tumour_loss,
    loss
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

CONDITIONAL LDM V4 PRE-FLIGHT DIAGNOSTIC

PART 1 — DIFFUSION SCHEDULE
Timesteps: 1000
First beta: 4.124641418457031e-05
Last beta: 1.0
First alpha_cumprod: 0.9999587535858154
Final alpha_cumprod: 0.0
Final SNR: 0.0
RESULT: schedule check PASSED.

PART 2 — VAE LATENT PIPELINE
Image shape: (1, 1, 208, 224, 160)
Raw latent shape: (1, 4, 52, 56, 40)
Normalized latent shape: (1, 4, 52, 56, 40)
Latent mask shape: (1, 3, 52, 56, 40)
Raw entropy: 7.373809814453125
Normalized entropy: 1.5167115926742554

Raw latent statistics
Channel 0: mean=-0.3140, std=0.6014
Channel 1: mean=-0.6394, std=0.6252
Channel 2: mean=0.7511, std=0.9685
Channel 3: mean=0.0193, std=0.2190

Normalized latent statistics
Channel 0: mean=-0.4155, std=0.8839
Channel 1: mean=-0.4698, std=0.6592
Channel 2: mean=0.4331, std=0.6810
Channel 3: mean=0.0226, std=1.0340

Tumour latent fraction: 0.02479395642876625
RESULT: latent pipeline PASSED.

PART 3 — v-PREDICTION ALGEBRA
t=  0 | x0 algebra L1=0.00000002
t= 50 | x0 algebra L1=

/tmp/ipykernel_3274405/4197186835.py:392: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


In [35]:
# ============================================================
# CONDITIONAL LDM V4 — TWO-STEP GRADIENT FLOW TEST
# ============================================================

import copy

print("=" * 72)
print("CONDITIONAL LDM V4 — TWO-STEP GRADIENT FLOW TEST")
print("=" * 72)

# Fresh model so previous smoke tests do not contaminate the result
test_model = ConditionalLatentUNet3D(
    latent_channels=4,
    base_channels=64,
    condition_dim=256,
    entropy_scale=0.1
).to(device)

test_optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

vae.eval()

latent_mean = LATENT_MEAN.to(device)
latent_std = LATENT_STD.to(device)


def get_grad_norm(parameter):
    if parameter.grad is None:
        return None
    return parameter.grad.detach().norm().item()


def run_test_step(step_number):

    test_model.train()

    batch = next(iter(train_loader))

    x = batch["image"].to(device)
    mask = batch["mask"].to(device)

    entropy = (
        batch["heterogeneity"]
        .to(device)
    )

    entropy = (
        entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD

    with torch.no_grad():

        mu, logvar = vae.encoder(x)

        z = (
            mu
            - latent_mean
        ) / latent_std

    latent_mask = prepare_latent_mask(
        mask
    )

    t = torch.randint(
        0,
        timesteps,
        (z.shape[0],),
        device=device,
        dtype=torch.long
    )

    noise = torch.randn_like(z)

    z_t = q_sample(
        z,
        t,
        noise
    )

    v_target = get_v_target(
        z,
        noise,
        t
    )

    test_optimizer.zero_grad(
        set_to_none=True
    )

    v_pred = test_model(
        z_t,
        t,
        latent_mask,
        entropy
    )

    squared_error = (
        v_pred
        - v_target
    ) ** 2

    global_loss = squared_error.mean()

    tumour_region = (
        latent_mask.sum(
            dim=1,
            keepdim=True
        )
        > 0
    )

    tumour_region = tumour_region.expand(
        -1,
        4,
        -1,
        -1,
        -1
    )

    if tumour_region.any():
        tumour_loss = (
            squared_error[
                tumour_region
            ].mean()
        )
    else:
        tumour_loss = torch.zeros(
            (),
            device=device
        )

    loss = (
        global_loss
        + 0.25 * tumour_loss
    )

    loss.backward()

    print()
    print(f"STEP {step_number}")

    print(
        "Loss:",
        loss.item()
    )

    print(
        "input_conv:",
        get_grad_norm(
            test_model.input_conv.weight
        )
    )

    print(
        "mask_level1:",
        get_grad_norm(
            test_model.mask_level1.weight
        )
    )

    print(
        "mask_level2:",
        get_grad_norm(
            test_model.mask_level2.weight
        )
    )

    print(
        "mask_level3:",
        get_grad_norm(
            test_model.mask_level3.weight
        )
    )

    print(
        "entropy final layer:",
        get_grad_norm(
            test_model
            .entropy_embedding[-1]
            .weight
        )
    )

    print(
        "out_conv:",
        get_grad_norm(
            test_model.out_conv.weight
        )
    )

    test_optimizer.step()


# Step 1
run_test_step(1)

# Step 2
run_test_step(2)

# Step 3 for extra certainty
run_test_step(3)


print()
print("=" * 72)
print("INTERPRETATION")
print("=" * 72)

print(
    "Expected:"
)

print(
    "Step 1: upstream gradients may be zero "
    "because out_conv starts at zero."
)

print(
    "Step 2/3: input_conv should become non-zero."
)

print(
    "Mask/entropy branches should also begin "
    "receiving non-zero gradients."
)

print(
    "If they remain exactly 0 through Step 3, "
    "do NOT start full training."
)


del test_model
del test_optimizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

CONDITIONAL LDM V4 — TWO-STEP GRADIENT FLOW TEST

STEP 1
Loss: 1.79874587059021
input_conv: 0.0
mask_level1: 0.0
mask_level2: 0.0
mask_level3: 0.0
entropy final layer: 0.0
out_conv: 18.198081970214844

STEP 2
Loss: 3.0009045600891113
input_conv: 0.13733844459056854
mask_level1: 0.022332025691866875
mask_level2: 0.00046220968943089247
mask_level3: 2.2845340936328284e-05
entropy final layer: 0.0
out_conv: 8.777382850646973

STEP 3
Loss: 2.7208356857299805
input_conv: 0.7567382454872131
mask_level1: 0.05088968575000763
mask_level2: 0.005752592347562313
mask_level3: 0.0004779620503541082
entropy final layer: 0.0
out_conv: 26.621109008789062

INTERPRETATION
Expected:
Step 1: upstream gradients may be zero because out_conv starts at zero.
Step 2/3: input_conv should become non-zero.
Mask/entropy branches should also begin receiving non-zero gradients.
If they remain exactly 0 through Step 3, do NOT start full training.


In [38]:
# ============================================================
# CONDITIONAL LDM V4 — FINAL INTEGRATED PRE-TRAINING CHECK
# ============================================================

import os
import copy

print("=" * 76)
print("CONDITIONAL LDM V4 — FINAL INTEGRATED PRE-TRAINING CHECK")
print("=" * 76)


# ------------------------------------------------------------
# Fresh temporary model
# Do NOT modify the real training model
# ------------------------------------------------------------

test_model = ConditionalLatentUNet3D(
    latent_channels=4,
    base_channels=64,
    condition_dim=256,
    entropy_scale=0.1
).to(device)

test_ema = EMA(
    test_model,
    decay=0.9999
)

test_optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

vae.eval()

latent_mean = LATENT_MEAN.to(device)
latent_std = LATENT_STD.to(device)


def grad_norm(parameter):

    if parameter.grad is None:
        return None

    return (
        parameter.grad
        .detach()
        .norm()
        .item()
    )


# ============================================================
# PART 1 — 10-STEP CONDITION GRADIENT TEST
# ============================================================

print()
print("=" * 76)
print("PART 1 — 10-STEP CONDITION GRADIENT TEST")
print("=" * 76)

entropy_activated = False
mask_activated = False
input_activated = False


for step in range(1, 11):

    test_model.train()

    batch = next(iter(train_loader))

    x = batch["image"].to(
        device,
        non_blocking=True
    )

    mask = batch["mask"].to(
        device,
        non_blocking=True
    )

    entropy = batch[
        "heterogeneity"
    ].to(
        device,
        non_blocking=True
    )

    entropy = (
        entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    with torch.no_grad():

        mu, logvar = vae.encoder(x)

        z = (
            mu
            - latent_mean
        ) / latent_std


    latent_mask = prepare_latent_mask(
        mask
    )


    t = torch.randint(
        0,
        timesteps,
        (z.shape[0],),
        device=device,
        dtype=torch.long
    )

    noise = torch.randn_like(z)

    z_t = q_sample(
        z,
        t,
        noise
    )

    v_target = get_v_target(
        z,
        noise,
        t
    )


    test_optimizer.zero_grad(
        set_to_none=True
    )


    v_pred = test_model(
        z_t,
        t,
        latent_mask,
        entropy
    )


    squared_error = (
        v_pred
        - v_target
    ) ** 2


    global_loss = (
        squared_error.mean()
    )


    tumour_region = (
        latent_mask.sum(
            dim=1,
            keepdim=True
        )
        > 0
    )

    tumour_region = tumour_region.expand(
        -1,
        4,
        -1,
        -1,
        -1
    )


    if tumour_region.any():

        tumour_loss = (
            squared_error[
                tumour_region
            ].mean()
        )

    else:

        tumour_loss = torch.zeros(
            (),
            device=device,
            dtype=global_loss.dtype
        )


    loss = (
        global_loss
        + 0.25 * tumour_loss
    )


    loss.backward()


    g_input = grad_norm(
        test_model.input_conv.weight
    )

    g_mask1 = grad_norm(
        test_model.mask_level1.weight
    )

    g_mask2 = grad_norm(
        test_model.mask_level2.weight
    )

    g_mask3 = grad_norm(
        test_model.mask_level3.weight
    )

    g_entropy = grad_norm(
        test_model
        .entropy_embedding[-1]
        .weight
    )

    g_out = grad_norm(
        test_model.out_conv.weight
    )


    print(
        f"Step {step:02d} | "
        f"Loss={loss.item():.5f} | "
        f"Input={g_input:.6e} | "
        f"Mask1={g_mask1:.6e} | "
        f"Mask2={g_mask2:.6e} | "
        f"Mask3={g_mask3:.6e} | "
        f"Entropy={g_entropy:.6e} | "
        f"Out={g_out:.6e}"
    )


    if (
        g_input is not None
        and g_input > 0
    ):
        input_activated = True

    if (
        g_mask1 is not None
        and g_mask1 > 0
    ):
        mask_activated = True

    if (
        g_entropy is not None
        and g_entropy > 0
    ):
        entropy_activated = True


    test_optimizer.step()

    test_ema.update(
        test_model
    )


print()

print(
    "Input gradient activated:",
    input_activated
)

print(
    "Mask gradient activated:",
    mask_activated
)

print(
    "Entropy gradient activated:",
    entropy_activated
)


# ============================================================
# PART 2 — EMA UPDATE CHECK
# ============================================================

print()
print("=" * 76)
print("PART 2 — EMA UPDATE CHECK")
print("=" * 76)


raw_weight = (
    test_model.out_conv.weight
    .detach()
)

ema_weight = (
    test_ema.ema_model
    .out_conv.weight
    .detach()
)


ema_difference = (
    raw_weight
    - ema_weight
).abs().mean().item()


ema_nonzero = (
    ema_weight.abs()
    .mean()
    .item()
)


print(
    "Raw vs EMA mean absolute difference:",
    ema_difference
)

print(
    "EMA weight mean absolute value:",
    ema_nonzero
)


assert ema_nonzero > 0

print(
    "RESULT: EMA is updating."
)


# ============================================================
# PART 3 — CHECKPOINT SAVE / LOAD (ROBUST VERSION)
# ============================================================

print()
print("=" * 76)
print("PART 3 — CHECKPOINT SAVE / LOAD")
print("=" * 76)


test_checkpoint_path = (
    "conditional_ldm_v4_"
    "temporary_test_checkpoint.pt"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

save_ldm_checkpoint(
    model=test_model,
    ema=test_ema,
    optimizer=test_optimizer,
    epoch=10,
    path=test_checkpoint_path
)

assert os.path.exists(
    test_checkpoint_path
)

print(
    "Temporary checkpoint saved."
)


# ------------------------------------------------------------
# New model / EMA / optimizer
# ------------------------------------------------------------

reload_model = ConditionalLatentUNet3D(
    latent_channels=4,
    base_channels=64,
    condition_dim=256,
    entropy_scale=0.1
).to(device)


reload_ema = EMA(
    reload_model,
    decay=0.9999
)


reload_optimizer = torch.optim.AdamW(
    reload_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

loaded_epoch = load_ldm_checkpoint(
    model=reload_model,
    ema=reload_ema,
    optimizer=reload_optimizer,
    path=test_checkpoint_path,
    device=device
)

print(
    "Loaded epoch:",
    loaded_epoch
)

assert loaded_epoch == 10


# ------------------------------------------------------------
# 1. Direct RAW model parameter comparison
# ------------------------------------------------------------

max_parameter_difference = 0.0

parameter_mismatch_count = 0


original_state = (
    test_model.state_dict()
)

reload_state = (
    reload_model.state_dict()
)


for key in original_state.keys():

    difference = (
        original_state[key]
        .detach()
        .float()
        .cpu()
        -
        reload_state[key]
        .detach()
        .float()
        .cpu()
    ).abs().max().item()

    max_parameter_difference = max(
        max_parameter_difference,
        difference
    )

    if difference != 0.0:
        parameter_mismatch_count += 1


print(
    "RAW model max parameter difference:",
    max_parameter_difference
)

print(
    "RAW model mismatched tensors:",
    parameter_mismatch_count
)


# ------------------------------------------------------------
# 2. Direct EMA parameter comparison
# ------------------------------------------------------------

max_ema_difference = 0.0

ema_mismatch_count = 0


original_ema_state = (
    test_ema.ema_model.state_dict()
)

reload_ema_state = (
    reload_ema.ema_model.state_dict()
)


for key in original_ema_state.keys():

    difference = (
        original_ema_state[key]
        .detach()
        .float()
        .cpu()
        -
        reload_ema_state[key]
        .detach()
        .float()
        .cpu()
    ).abs().max().item()

    max_ema_difference = max(
        max_ema_difference,
        difference
    )

    if difference != 0.0:
        ema_mismatch_count += 1


print(
    "EMA max parameter difference:",
    max_ema_difference
)

print(
    "EMA mismatched tensors:",
    ema_mismatch_count
)


# These are the important checkpoint checks
assert max_parameter_difference == 0.0
assert parameter_mismatch_count == 0

assert max_ema_difference == 0.0
assert ema_mismatch_count == 0


print(
    "Parameter save/reload: EXACT MATCH."
)


# ------------------------------------------------------------
# 3. Forward numerical comparison
# ------------------------------------------------------------

test_model.eval()
reload_model.eval()


with torch.no_grad():

    original_output = test_model(
        z_t,
        t,
        latent_mask,
        entropy
    )

    reload_output = reload_model(
        z_t,
        t,
        latent_mask,
        entropy
    )


reload_difference_max = (
    original_output
    - reload_output
).abs().max().item()


reload_difference_mean = (
    original_output
    - reload_output
).abs().mean().item()


print(
    "Forward max absolute difference:",
    reload_difference_max
)

print(
    "Forward mean absolute difference:",
    reload_difference_mean
)


# CUDA attention can introduce very small numerical differences.
assert reload_difference_mean < 1e-5
assert reload_difference_max < 1e-4


print(
    "RESULT: checkpoint save/reload PASSED."
)


# ============================================================
# PART 4 — 3-STEP REVERSE SAMPLER SANITY CHECK
# ============================================================

print()
print("=" * 76)
print("PART 4 — REVERSE SAMPLER 3-STEP CHECK")
print("=" * 76)


sampler_model = (
    reload_ema.ema_model
)

sampler_model.eval()


sample_x = torch.randn(
    (
        1,
        4,
        52,
        56,
        40
    ),
    device=device
)


coef1 = (
    posterior_mean_coef1
    .to(device)
)

coef2 = (
    posterior_mean_coef2
    .to(device)
)

posterior_var = (
    posterior_variance
    .to(device)
)


sampler_steps = [
    999,
    998,
    997
]


with torch.no_grad():

    for step in sampler_steps:

        sampler_t = torch.full(
            (1,),
            step,
            device=device,
            dtype=torch.long
        )


        v_pred_step = sampler_model(
            sample_x,
            sampler_t,
            latent_mask,
            entropy
        )


        x0_pred = v_to_x0(
            sample_x,
            v_pred_step,
            sampler_t
        )


        x0_pred = torch.clamp(
            x0_pred,
            -5.0,
            5.0
        )


        model_mean = (
            coef1[step]
            * x0_pred
            +
            coef2[step]
            * sample_x
        )


        if step > 0:

            step_noise = (
                torch.randn_like(
                    sample_x
                )
            )

            sample_x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[step]
                )
                * step_noise
            )

        else:

            sample_x = model_mean


        finite = (
            torch.isfinite(
                sample_x
            ).all().item()
        )


        print(
            f"Step {step} | "
            f"mean={sample_x.mean().item():.5f} | "
            f"std={sample_x.std().item():.5f} | "
            f"min={sample_x.min().item():.5f} | "
            f"max={sample_x.max().item():.5f} | "
            f"finite={finite}"
        )


        assert finite


print(
    "RESULT: reverse sampler remains finite."
)


# ============================================================
# PART 5 — LATENT DENORMALIZATION + VAE DECODE
# ============================================================

print()
print("=" * 76)
print("PART 5 — LATENT DENORMALIZATION + VAE DECODE")
print("=" * 76)


with torch.no_grad():

    decoded_latent = (
        sample_x
        * latent_std
        + latent_mean
    )


    decoded_image = vae.decoder(
        decoded_latent
    )


print(
    "Decoded latent shape:",
    tuple(
        decoded_latent.shape
    )
)

print(
    "Decoded MRI shape:",
    tuple(
        decoded_image.shape
    )
)

print(
    "Decoded MRI range:",
    decoded_image.min().item(),
    decoded_image.max().item()
)

print(
    "Decoded MRI mean:",
    decoded_image.mean().item()
)

print(
    "Decoded MRI std:",
    decoded_image.std().item()
)


assert (
    decoded_image.shape
    == (
        1,
        1,
        208,
        224,
        160
    )
)

assert torch.isfinite(
    decoded_image
).all()


print(
    "RESULT: latent → VAE decode pipeline PASSED."
)


# ============================================================
# PART 6 — FINAL SUMMARY
# ============================================================

print()
print("=" * 76)
print("FINAL PRE-TRAINING SUMMARY")
print("=" * 76)


print(
    "Input backbone gradient:",
    "PASS"
    if input_activated
    else "FAIL"
)

print(
    "Tumour-mask gradient:",
    "PASS"
    if mask_activated
    else "FAIL"
)

print(
    "Entropy gradient:",
    "PASS"
    if entropy_activated
    else "FAIL"
)

print(
    "EMA update: PASS"
)

print(
    "Checkpoint save/reload: PASS"
)

print(
    "Reverse sampler finite: PASS"
)

print(
    "Latent denormalization + decoder: PASS"
)


if (
    input_activated
    and mask_activated
    and entropy_activated
):

    print()
    print(
        "FINAL RESULT:"
    )

    print(
        "Conditional LDM V4 is ready "
        "for full 50-epoch training."
    )

else:

    print()
    print(
        "FINAL RESULT:"
    )

    print(
        "DO NOT start full training yet."
    )

    print(
        "At least one conditioning/"
        "gradient path still needs correction."
    )


# ============================================================
# CLEANUP
# ============================================================

if os.path.exists(
    test_checkpoint_path
):
    os.remove(
        test_checkpoint_path
    )


del test_model
del test_ema
del test_optimizer

del reload_model
del reload_ema
del reload_optimizer

del decoded_latent
del decoded_image
del sample_x


if torch.cuda.is_available():
    torch.cuda.empty_cache()

CONDITIONAL LDM V4 — FINAL INTEGRATED PRE-TRAINING CHECK

PART 1 — 10-STEP CONDITION GRADIENT TEST
Step 01 | Loss=1.49380 | Input=0.000000e+00 | Mask1=0.000000e+00 | Mask2=0.000000e+00 | Mask3=0.000000e+00 | Entropy=0.000000e+00 | Out=2.105735e+01
Step 02 | Loss=1.33556 | Input=1.724247e-01 | Mask1=5.099023e-03 | Mask2=1.074540e-04 | Mask3=5.239315e-06 | Entropy=0.000000e+00 | Out=1.123991e+01
Step 03 | Loss=1.20818 | Input=4.127462e-02 | Mask1=7.149987e-04 | Mask2=5.138413e-05 | Mask3=8.866140e-06 | Entropy=0.000000e+00 | Out=2.902851e+00
Step 04 | Loss=1.93864 | Input=1.208269e+00 | Mask1=9.918536e-02 | Mask2=9.814095e-03 | Mask3=4.852716e-04 | Entropy=1.161279e-04 | Out=4.624150e+01
Step 05 | Loss=1.21088 | Input=1.443271e-01 | Mask1=1.322475e-02 | Mask2=1.725338e-03 | Mask3=1.103756e-04 | Entropy=9.768203e-06 | Out=1.208093e+01
Step 06 | Loss=1.01000 | Input=1.992877e-01 | Mask1=2.990709e-02 | Mask2=4.025893e-03 | Mask3=1.981284e-04 | Entropy=3.926754e-05 | Out=1.949067e+01
Step 07